# Adding more data

We already went through all of these steps so we will just show the code without much explanation

In [215]:
import pandas as pd
import warnings

warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*un-recognized timezone.*"
)

In [216]:
df_abc = pd.read_csv("data/news/original/abc_headlines.csv")


In [217]:
df_abc.head()

,publish_date,headline_text
0,20030219,aba decides against community broadcasting lic...
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers


In [218]:
df_abc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1244184 entries, 0 to 1244183
Data columns (total 2 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   publish_date   1244184 non-null  int64 
 1   headline_text  1244184 non-null  object
dtypes: int64(1), object(1)
memory usage: 19.0+ MB


In [219]:
df_abc.dropna(inplace=True)
df_abc.reset_index(drop=True, inplace=True)

In [220]:
df_abc.head()

,publish_date,headline_text
0,20030219,aba decides against community broadcasting lic...
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers


In [221]:
df_abc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1244184 entries, 0 to 1244183
Data columns (total 2 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   publish_date   1244184 non-null  int64 
 1   headline_text  1244184 non-null  object
dtypes: int64(1), object(1)
memory usage: 19.0+ MB


In [222]:
df_abc["date"] = pd.to_datetime(
    df_abc["publish_date"], format="%Y%m%d", errors="coerce"
)
min_date_abc = df_abc["date"].min()
max_date_abc = df_abc["date"].max()
print(min_date_abc)
print(max_date_abc)

2003-02-19 00:00:00
2021-12-31 00:00:00


In [223]:
df_abc.rename(columns={"headline_text": "summary"}, inplace=True)

In [224]:
df_abc["source"] = "abc"

In [225]:
df_abc[["date", "summary", "source"]].to_csv("data/news/filtered/abc_filtered.csv", index=False)

In [226]:
df_abc["adj_date"] = df_abc["date"]

# Saturday posts get moved +2 days (to Monday)
df_abc.loc[df_abc["adj_date"].dt.weekday == 5, "adj_date"] += pd.Timedelta(days=2)

# Sunday posts get moved +1 day (to Monday)
df_abc.loc[df_abc["adj_date"].dt.weekday == 6, "adj_date"] += pd.Timedelta(days=1)

In [227]:
df_abc["adj_date"] = df_abc["adj_date"].dt.normalize()

In [228]:
df_abc.to_csv("data/news/filtered/abc_filtered.csv", index=False)
df_abc.head()

,publish_date,summary,date,source,adj_date
0,20030219,aba decides against community broadcasting lic...,2003-02-19,abc,2003-02-19
1,20030219,act fire witnesses must be aware of defamation,2003-02-19,abc,2003-02-19
2,20030219,a g calls for infrastructure protection summit,2003-02-19,abc,2003-02-19
3,20030219,air nz staff in aust strike for pay rise,2003-02-19,abc,2003-02-19
4,20030219,air nz strike to affect australian travellers,2003-02-19,abc,2003-02-19


In [229]:
df_abc = df_abc[["summary", "adj_date"]]

df_abc.head()

,summary,adj_date
0,aba decides against community broadcasting lic...,2003-02-19
1,act fire witnesses must be aware of defamation,2003-02-19
2,a g calls for infrastructure protection summit,2003-02-19
3,air nz staff in aust strike for pay rise,2003-02-19
4,air nz strike to affect australian travellers,2003-02-19


In [230]:
%%capture
%pip install nltk

In [231]:
import pandas as pd
import numpy as np
import re

In [232]:
%%capture
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")  # for lemmatization

In [233]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

finance_whitelist = {
    "up",
    "down",
    "above",
    "below",
    "under",
    "over",
    "rise",
    "fall",
    "not",
    "no",
    "nor",
    "neither",
    "never",
    "none",
    "more",
    "most",
    "few",
    "less",
}

stop_words.difference_update(finance_whitelist)

In [234]:
def clean_text(text):
    # 1. Lowercase all text
    text = text.lower()
    # 2. Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # 3. Tokenize using Punkt
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # 5. Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [235]:
df_abc["clean_summary"] = df_abc["summary"].apply(clean_text)

df_abc.head()

,summary,adj_date,clean_summary
0,aba decides against community broadcasting lic...,2003-02-19,aba decides community broadcasting licence
1,act fire witnesses must be aware of defamation,2003-02-19,act fire witness must aware defamation
2,a g calls for infrastructure protection summit,2003-02-19,g call infrastructure protection summit
3,air nz staff in aust strike for pay rise,2003-02-19,air nz staff aust strike pay rise
4,air nz strike to affect australian travellers,2003-02-19,air nz strike affect australian traveller


In [236]:
df_abc_clean = df_abc[["adj_date", "clean_summary"]].copy()

df_abc_clean.rename(
    columns={"adj_date": "date", "clean_summary": "summary"}, inplace=True
)

df_abc_clean.head()

,date,summary
0,2003-02-19,aba decides community broadcasting licence
1,2003-02-19,act fire witness must aware defamation
2,2003-02-19,g call infrastructure protection summit
3,2003-02-19,air nz staff aust strike pay rise
4,2003-02-19,air nz strike affect australian traveller


One thing I forgot to do was to put all the text into one row for where the date is the same.

In [237]:
df_abc_grouped = df_abc_clean.groupby("date")["summary"].apply(" ".join).reset_index()
df_abc_grouped.head()

,date,summary
0,2003-02-19,aba decides community broadcasting licence act...
1,2003-02-20,15 dead rebel bombing raid philippine army aba...
2,2003-02-21,accc timid petrol price investigation action w...
3,2003-02-24,86 confirmed dead u nightclub fire act tourist...
4,2003-02-25,4 million pay sacked ceo aid organisation disa...


In [238]:
df_abc_grouped.to_csv("data/news/cleaned/abc_clean.csv", index=False)

df_abc_grouped.head()

,date,summary
0,2003-02-19,aba decides community broadcasting licence act...
1,2003-02-20,15 dead rebel bombing raid philippine army aba...
2,2003-02-21,accc timid petrol price investigation action w...
3,2003-02-24,86 confirmed dead u nightclub fire act tourist...
4,2003-02-25,4 million pay sacked ceo aid organisation disa...


In [239]:
%%capture
%pip install yfinance

In [240]:
import pandas as pd
import warnings

warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*un-recognized timezone.*"
)


In [241]:
import yfinance as yf

# this is a sunday that is why there is no finicial data on it
start_date = "2003-02-19" # 2003-02-19 extra dateset
end_date = "2021-12-31" # 2021-12-31 extra dataset

raw_data = yf.download(tickers="^GSPC", start=start_date, end=end_date, interval="1d")

# flattening multi-level columns
raw_data.columns = [
    "_".join([str(c) for c in col if c != ""]) if isinstance(col, tuple) else col
    for col in raw_data.columns
]
# creating date a column from index
raw_data = raw_data.reset_index().rename(columns={"index": "Date"})

C:\Users\vince\AppData\Local\Temp\ipykernel_26668\143102806.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers="^GSPC", start=start_date, end=end_date, interval="1d")
[*********************100%***********************]  1 of 1 completed


In [242]:
raw_data.head()

,Date,Close_^GSPC,High_^GSPC,Low_^GSPC,Open_^GSPC,Volume_^GSPC
0,2003-02-19,845.130005,851.169983,838.789978,851.169983,1075600000
1,2003-02-20,837.099976,849.369995,836.559998,845.130005,1194100000
2,2003-02-21,848.169983,852.280029,831.479980,837.099976,1398200000
3,2003-02-24,832.580017,848.169983,832.159973,848.169983,1229200000
4,2003-02-25,838.570007,839.549988,818.539978,832.580017,1483700000


In [243]:
raw_data.info()
# there is no null data in the dataset
raw_data.isnull().sum()

# creating a copy and saving it for better name clarity
df_stock = raw_data.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4751 entries, 0 to 4750
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          4751 non-null   datetime64[ns]
 1   Close_^GSPC   4751 non-null   float64       
 2   High_^GSPC    4751 non-null   float64       
 3   Low_^GSPC     4751 non-null   float64       
 4   Open_^GSPC    4751 non-null   float64       
 5   Volume_^GSPC  4751 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 222.8 KB


In [244]:
# calculating percent change for closing price
df_stock["pct_change"] = df_stock["Close_^GSPC"].pct_change()

# Shifting tomorrows close
df_stock["Close_T+1"] = df_stock["Close_^GSPC"].shift(-1)

# predict whether tomorrow will go UP
df_stock["Target"] = (df_stock["Close_T+1"] > df_stock["Close_^GSPC"]).astype(int)

# selecting relevant columns for our testing and training and merging into new dataframe
df_stock_refined = df_stock[["Volume_^GSPC", "pct_change", "Target", "Date"]].copy()

# rename columns to lowercase because im a jerk
df_stock_refined.columns = ["volume", "pct_change", "target", "date"]

# removing the first row with NaN pct_change
# also remove the last row with NaN Target
df_stock_refined = df_stock_refined.dropna().reset_index(drop=True)

df_stock_refined.head()

,volume,pct_change,target,date
0,1194100000,-0.009502,1,2003-02-20
1,1398200000,0.013224,0,2003-02-21
2,1229200000,-0.018381,1,2003-02-24
3,1483700000,0.007194,0,2003-02-25
4,1374400000,-0.013141,1,2003-02-26


In [245]:
import pandas as pd

news_data = df_abc_grouped.copy()

news_data = news_data.sort_values(by="date", ascending=True).reset_index(drop=True)

news_data.head()

,date,summary
0,2003-02-19,aba decides community broadcasting licence act...
1,2003-02-20,15 dead rebel bombing raid philippine army aba...
2,2003-02-21,accc timid petrol price investigation action w...
3,2003-02-24,86 confirmed dead u nightclub fire act tourist...
4,2003-02-25,4 million pay sacked ceo aid organisation disa...


In [246]:
yfinance_data = df_stock_refined.copy()

# sort by date ascending
yfinance_data = yfinance_data.sort_values(by="date", ascending=True).reset_index(
    drop=True
)
yfinance_data.head()

,volume,pct_change,target,date
0,1194100000,-0.009502,1,2003-02-20
1,1398200000,0.013224,0,2003-02-21
2,1229200000,-0.018381,1,2003-02-24
3,1483700000,0.007194,0,2003-02-25
4,1374400000,-0.013141,1,2003-02-26


In [247]:
# Combining datasets based on time outer join so that it keeps all data from both datasets
merged_data = pd.merge(news_data, yfinance_data, on='date', how='outer')

In [248]:
merged_data[merged_data['summary'].isnull() | (merged_data['summary'] == '')]

,date,summary,volume,pct_change,target
326,2004-05-20,NaN,1.211000e+09,0.000468,1.0
400,2004-09-01,NaN,1.142100e+09,0.001512,1.0
572,2005-04-29,NaN,2.362360e+09,0.011922,1.0
964,2006-10-31,NaN,2.803030e+09,0.000007,0.0
1227,2007-11-02,NaN,4.285990e+09,0.000802,0.0
4407,2020-01-10,NaN,3.214580e+09,-0.002855,1.0


In [249]:
merged_data = merged_data[merged_data['summary'].notnull()].reset_index(drop=True)

In [250]:
# removing all data points where there is no news summary
merged_data[merged_data['summary'].isnull() | (merged_data['summary'] == '')]

,date,summary,volume,pct_change,target


In [251]:
merged_data[merged_data["volume"].isnull()]

,date,summary,volume,pct_change,target
0,2003-02-19,aba decides community broadcasting licence act...,NaN,NaN,NaN
42,2003-04-18,afl defends collingwood banner censorship set ...,NaN,NaN,NaN
68,2003-05-26,10 pro democracy activist jailed burma 14 russ...,NaN,NaN,NaN
97,2003-07-04,abalone industry picking up sars academy build...,NaN,NaN,NaN
138,2003-09-01,10 injured minibus crash abuse hotline program...,NaN,NaN,NaN
...,...,...,...,...,...
4787,2021-07-05,brad hazzard issue stern rebuke anti mask view...,NaN,NaN,NaN
4832,2021-09-06,afghanistan brink economic collapse dr paul gr...,NaN,NaN,NaN
4890,2021-11-25,adelaide red turn back betting sponsorship and...,NaN,NaN,NaN
4911,2021-12-24,40 million crystal meth destined australia aus...,NaN,NaN,NaN


In [252]:
import numpy as np

# 1. Create a "grouper" column
# We identify valid trading days. If volume exists, we keep the date.
# If volume is NaN, we set it to NaT (Not a Time) so we can fill it later.
merged_data["trading_day_group"] = merged_data["date"].where(
    merged_data["volume"].notnull(), pd.NaT
)

# 2. Backfill the dates
# take the date of the next valid row and pulls it UP into the previous NaN rows
merged_data["trading_day_group"] = merged_data["trading_day_group"].bfill()

# 3. Group by this new column and aggregate
# We combine the summaries and keep the financial data from the valid trading day (the last entry in the group)
shifted_data = (
    merged_data.groupby("trading_day_group")
    .agg(
        {
            "date": "last",  # Keep the actual trading date
            "summary": " ".join,  # Join the news strings together with a space
            "volume": "last",  # Take the volume from the valid day
            "pct_change": "last",  # Take the change from the valid day
            "target": "last",  # Take the target from the valid day
        }
    )
    .reset_index(drop=True)
)

# Remove any rows that remained NaN
shifted_data = shifted_data.dropna(subset=["volume"])

In [253]:
shifted_data[shifted_data["volume"].isnull()]

,date,summary,volume,pct_change,target


In [254]:
shifted_data['pct_change_lag1'] = shifted_data['pct_change'].shift(1)
shifted_data['volume_lag1'] = shifted_data['volume'].shift(1)
shifted_data['volume_5d_avg'] = shifted_data['volume'].rolling(window=5).mean()

In [255]:
shifted_data.to_csv('data/extra_and_merged_data.csv', index=False)
shifted_data.head()

,date,summary,volume,pct_change,target,pct_change_lag1,volume_lag1,volume_5d_avg
0,2003-02-20,aba decides community broadcasting licence act...,1.194100e+09,-0.009502,1.0,NaN,NaN,NaN
1,2003-02-21,accc timid petrol price investigation action w...,1.398200e+09,0.013224,0.0,-0.009502,1.194100e+09,NaN
2,2003-02-24,86 confirmed dead u nightclub fire act tourist...,1.229200e+09,-0.018381,1.0,0.013224,1.398200e+09,NaN
3,2003-02-25,4 million pay sacked ceo aid organisation disa...,1.483700e+09,0.007194,0.0,-0.018381,1.229200e+09,NaN
4,2003-02-26,ab defends nt population shrinking claim acb r...,1.374400e+09,-0.013141,1.0,0.007194,1.483700e+09,1.335920e+09
